# RAG 接続の実機確認（Issue #7・Colab 用）

v2 の同期パイプラインから v1 の検索（実 Qdrant）と生成（実 vLLM）を呼び、次を確かめる。データは架空の社内規程（evaluation/corpus）だけ。トークン不要（使うモデルは全て公開）。ランタイムは **T4 GPU**（無料枠）。有料枠（Google AI Pro 等）なら **L4** も選べる。4bit の計算部品は vLLM 0.32.0 の原文では T4（compute 7.5）・L4（8.9）・本番の RTX 3050（8.6）のどれでも同じ Marlin になる（Cutlass・Machete は 9.0 以上が要るので選ばれない）。T4 で残る未確認は attention の部品で、L4 は本番と同じ 8.0 以上の世代。dtype は本番の compose と同じ auto（このモデルは設定が float16 なので、どの GPU でも half で動く。実際の dtype は vLLM のログから結果に記録する）。

1. 2 問続けて回答できる（共有ループの使い回し）
2. 権限外の文書が出ない（v1 の生の検索結果を記録して照合）／2b 上位ユーザーには上位文書が出る（対照）
3. 監査に本文が残らない（監査 DB の行とファイルの生バイトの両方を見る）
4. vLLM・Qdrant を固める／落とすと保留で止まり、戻せば回答に戻る
5. 遅延（参考値。Colab の GPU は本番の 6GB 機と違うので合否には使わない）

2 回流す（合否の条件と検索 30 秒・生成 180 秒の上限はどちらも同じ）:

- **B（本番の既定・リランク on）**: 1〜3 と 5。4 は飛ばす。埋め込み・リランカーは v1 と同じく CPU で動く。Mac（M 系）でもリランク 30 件に 1 スレッド約 14 秒かかったので、Colab の CPU では検索が 30 秒を超えて保留になりうる。そうなっても試験の不具合ではなく「CPU のリランクが遅い」という結果として記録する
- **A（Issue #7 の合否・リランク off）**: 1〜5 全部。検索を速くして、vLLM を止める試験が検索の時間切れと混ざらないようにする。最後に両サーバーを落とすので、この後サーバーは止まっている

- 「Run all」で全セルを流す。結果のセルに A・B それぞれの `総合 PASS/FAIL` と結果 JSON（本文なし）が出て、その次のセルに vLLM が実際に選んだ部品（4bit の計算部品・attention）の行が出る
- B で 1 や 2 が retrieve で保留（TimeoutError）になるのは CPU のリランクが遅いため。モデルの問題ではないので MODEL は替えない。最初の時間切れの後も打ち切られたリランクが裏で CPU を使い続けるので、2 問目以降の retrieve の秒数は実際より遅く出る（独立した値は最初の 1 回だけ）
- vLLM が起動しないとき、または A の 1 が generate で保留（空の回答）になるときは、まず vLLM のログの末尾を Issue #7 に貼る（Gemma で何が起きたかの記録）。そのうえで:
  - ログに attention backend・unsupported GPU の話が出ているときは GPU と vLLM の相性の問題。L4 が使えるなら L4 で最初から流し直す。使えないなら、予備も同じエンジンなので MODEL は替えずに止める
  - ログに `Failed to find a kernel that can implement the WNA16 linear layer` が出ているときは、続く `Reasons:` の中の `MarlinLinearKernel` の行を見る（Cutlass・Machete の `requires capability 90` は T4 では毎回出るので無視する）。Marlin が `requires capability` なら入った vLLM が 0.32.0 と違う（0.32.0 では T4 でも Marlin が通る）ので、L4 が使えるなら L4 で最初から流し直す。Marlin が `Quant type ... not supported` や `Group size ... not supported` ならどの GPU でも同じ Marlin なので L4 でも解けない。下の予備に替える
  - それ以外（generate の保留など）は設定セルの `MODEL` を予備に替えて設定セルを実行し、最後のセルの下にある「やり直し」セルを実行してから vLLM のセル以降を流す（Qdrant のセルは流し直さない）。#7 は接続の試験なので予備で合否を出してよいが、Gemma は「未確認」として残す（本番も half で動くので、Gemma の数値の崩れは Colab だけの問題ではない）

In [ ]:
# 依存の衝突を避けるため、vLLM と RAG 側を別の仮想環境に入れる
!pip -q install uv
!git clone -q --depth 1 -b redesign/v2 https://github.com/Maaa2005/local-rag-agent.git repo
!uv venv -q /content/rag-env
# torch は CPU 版を先に入れる（同じ行で CPU の index を足すと uv が他の依存を解けない）
!uv pip install -q --python /content/rag-env/bin/python torch --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /content/rag-env/bin/python "transformers<5" sentence-transformers==3.3.1 \
    qdrant-client==1.12.1 aiosqlite==0.20.0 pydantic-settings==2.6.1 openai==1.57.0 httpx==0.28.1 pyyaml
!uv venv -q /content/vllm-env && uv pip install -q --python /content/vllm-env/bin/python vllm

In [ ]:
MODEL = "cyankiwi/gemma-4-E2B-it-AWQ-INT4"   # 本番の既定
# 名前は AWQ だが中身は compressed-tensors 形式（config.json の quant_method）。vLLM が自動で読むので
# --quantization は付けない（awq を付けると形式の不一致で起動前に落ちる）
EXTRA_ARGS = []
# 予備（Gemma 4 の 4bit が動かない場合）:
# MODEL = "sbintuitions/sarashina2.2-3b-instruct-v0.1"; EXTRA_ARGS = []
GPU_UTIL = 0.55          # 埋め込み等は CPU なので GPU は vLLM だけが使う
FAULT_TIMEOUT = 15       # 止める試験で待つ秒数
QDRANT_VER = "v1.12.6"   # qdrant-client 1.12.1 に合わせる

In [ ]:
import subprocess, time, urllib.request, pathlib, os, re

def launch(argv, log, env=None):
    # 新しいプロセスグループで起動する（止める試験で killpg を使うため。pid = pgid）
    f = open(log, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)

def wait(url, proc, log, limit):
    t = time.time()
    while time.time() - t < limit:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(url, timeout=3)
            print("ready:", url, f"{time.time() - t:.0f} 秒")
            return
        except Exception:
            time.sleep(3)
    # 末尾は巻き添えのエラー（vLLM なら API サーバー側の Traceback）で埋まり、本当の原因はもっと前に出る。
    # エラーらしい行の一覧と、最初のエラーから先、末尾の 3 つを出す（進捗バーの長い行は切る）
    lines = [l[:400] for l in pathlib.Path(log).read_text(errors="replace").splitlines()]
    hits = [i for i, l in enumerate(lines) if re.search(r"Traceback|ERROR|Error|Exception|Killed", l)]
    print("--- エラーらしい行（行番号: 中身）---")
    print("\n".join(f"{i + 1}: {lines[i][:300]}" for i in hits[:40]) or "なし")
    if hits:
        print(f"--- 最初のエラー（{hits[0] + 1} 行目）から 150 行 ---")
        print("\n".join(lines[hits[0]:hits[0] + 150]))
    print("--- ログの末尾 30 行 ---")
    print("\n".join(lines[-30:]))
    raise RuntimeError(f"起動しない: {url}（上はログの抜き出し・全文は {log}）")

qd = pathlib.Path("/content/qdrant"); (qd / "bin").mkdir(parents=True, exist_ok=True)
!curl -sSL https://github.com/qdrant/qdrant/releases/download/{QDRANT_VER}/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C /content/qdrant/bin
q_env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH=str(qd / "storage"), QDRANT__SERVICE__HTTP_PORT="6333")
qdrant = launch([str(qd / "bin" / "qdrant")], "/content/qdrant.log", q_env)
wait("http://127.0.0.1:6333/readyz", qdrant, "/content/qdrant.log", 60)

In [ ]:
import re
# GPU の名前と compute capability は記録用（dtype は本番の compose と同じ auto）
GPU_NAME, CAP = [x.strip() for x in subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
                                                   capture_output=True, text=True).stdout.splitlines()[0].split(",")]
GPU_TAG = GPU_NAME.split()[-1]
# 入った vLLM の版（setup は版を固定しないので記録する。計算部品の根拠は 0.32.0 の原文）
VLLM_VER = subprocess.run(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                          capture_output=True, text=True).stdout.strip() or "不明"
print(GPU_NAME, "compute", CAP, "vllm", VLLM_VER)
vllm = launch(["/content/vllm-env/bin/vllm", "serve", MODEL, "--served-model-name", "local-llm", "--port", "8001",
               "--dtype", "auto", "--max-model-len", "4096", "--gpu-memory-utilization", str(GPU_UTIL), *EXTRA_ARGS],
              "/content/vllm.log")
wait("http://127.0.0.1:8001/v1/models", vllm, "/content/vllm.log", 1200)
# auto が実際にどの dtype になったかをログから取る
m = re.search(r"dtype=torch\.(\w+)", pathlib.Path("/content/vllm.log").read_text())
DTYPE = m.group(1) if m else "不明（vllm.log を見る）"
print("dtype:", DTYPE)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# B: 本番の既定（リランク on）。止める試験はしない
CHECK = "repo/evaluation/colab/rag_connection_check.py"
WORK_B = f"/content/rag_work_B_{int(time.time())}"   # やり直しのたびに新しい作業場所にする
!/content/rag-env/bin/python {CHECK} --repo repo --work {WORK_B} --rerank on --label colab-{GPU_TAG}-rerank-on

In [ ]:
# A: Issue #7 の合否（リランク off）。止める試験で両サーバーを止めて戻し、最後に落とす
WORK_A = f"/content/rag_work_A_{int(time.time())}"
!/content/rag-env/bin/python {CHECK} --repo repo --work {WORK_A} \
    --qdrant-pgid {qdrant.pid} --vllm-pgid {vllm.pid} --fault-timeout {FAULT_TIMEOUT} --rerank off --label colab-{GPU_TAG}-rerank-off

In [ ]:
import json
for name, w in (("A（合否）", WORK_A), ("B（リランク on）", WORK_B)):
    f = pathlib.Path(w) / "rag_check_result.json"
    if not f.exists():
        print(name, "結果ファイルが無い（途中で落ちた。上のセルの出力を見る）:", f)
        continue
    r = json.loads(f.read_text())
    r["model"] = MODEL; r["gpu"] = GPU_NAME; r["compute_cap"] = CAP; r["dtype"] = DTYPE; r["vllm"] = VLLM_VER
    print(name, "総合", "PASS" if r.get("ok") else "FAIL")
    print(json.dumps(r, ensure_ascii=False, indent=1))

In [ ]:
# vLLM が実際に選んだ部品を記録する（Issue #7 の未確認: 4bit の計算部品が Marlin か・T4 の attention の部品）
keys = re.compile(r"mixed-precision linear|[Aa]ttention backend|Using .*[Bb]ackend|Marlin|dtype=")
hits = [l[:300] for l in pathlib.Path("/content/vllm.log").read_text(errors="replace").splitlines() if keys.search(l)]
print("\n".join(hits[:30]) if hits else "該当行なし（vllm.log を見る）")

## やり直し（モデルを替えるときだけ）
残っている vLLM と Qdrant を止め、Qdrant を起動し直す。この後 vLLM のセルから流す。

In [ ]:
import signal
for p in (globals().get("vllm"), globals().get("qdrant")):
    if p is not None:
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
time.sleep(5)
qdrant = launch([str(qd / "bin" / "qdrant")], "/content/qdrant.log", q_env)
wait("http://127.0.0.1:6333/readyz", qdrant, "/content/qdrant.log", 60)
!nvidia-smi --query-gpu=memory.used --format=csv